# Financial Risk Clustering

In [6]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Configuration

In [7]:
DATASET_PATH = 'archive/Financial Risk Classification Dataset.csv'
LABELED_OUTPUT_PATH = 'archive/financial_risk_labeled.csv'

FEATURES_TO_KEEP = [
    'loan_to_income_ratio', 'expenses_to_income_ratio', 'savings_to_income_ratio',
    'debt_to_income_ratio', 'credit_score', 'previous_default_count',
    'loan_duration_months', 'interest_rate', 'age', 'employment_stability_years'
]

## 2. Feature Engineering

In [8]:
def prepare_features(df):
    df = df.copy()
    df['annual_income'] = df['annual_income'].replace(0, 1)

    df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income']
    df['expenses_to_income_ratio'] = df['monthly_expenses'] / (df['annual_income'] / 12)
    df['savings_to_income_ratio'] = df['savings_balance'] / df['annual_income']

    missing_columns = [column for column in FEATURES_TO_KEEP if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    X = df[FEATURES_TO_KEEP].values
    return df, X

## 3. K-Means Label Generation

In [9]:
def find_optimal_k(X, min_k=3, max_k=5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    best_k = min_k
    best_score = -1
    silhouette_scores = {}

    print('Evaluating Silhouette Scores:')
    for k in range(min_k, max_k + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        silhouette_scores[k] = score
        print(f' - K={k}: Silhouette Score = {score:.4f}')

        if score > best_score:
            best_score = score
            best_k = k

    print(f'-> Optimal K based on Silhouette Score: {best_k}')
    return best_k, X_scaled, silhouette_scores


def generate_risk_labels(X_scaled, best_k):
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_scaled)

    centers = kmeans.cluster_centers_
    risk_scores = centers[:, 3] + centers[:, 5] - centers[:, 4]
    sorted_cluster_indices = np.argsort(risk_scores)
    cluster_mapping = {
        old_label: new_label
        for new_label, old_label in enumerate(sorted_cluster_indices)
    }

    y = np.array([cluster_mapping[label] for label in cluster_labels])
    return y, kmeans, cluster_mapping

## 4. Execution

In [10]:
print('Loading data...')
df_raw = pd.read_csv(DATASET_PATH)

print('Feature Engineering...')
df_features, X = prepare_features(df_raw)

print('\nStarting K-Means Clustering to generate Risk Profile labels...')
optimal_k, X_scaled, silhouette_scores = find_optimal_k(X)
risk_labels, kmeans_model, cluster_mapping = generate_risk_labels(X_scaled, optimal_k)

df_labeled = df_features.copy()
df_labeled['risk_profile_label'] = risk_labels

print('\nLabel Distribution setelah Clustering:')
print(pd.Series(risk_labels).value_counts().sort_index())
print(f'Cluster mapping: {cluster_mapping}')

df_labeled.to_csv(LABELED_OUTPUT_PATH, index=False)
print(f'\nLabeled data saved to: {LABELED_OUTPUT_PATH}')

Loading data...
Feature Engineering...

Starting K-Means Clustering to generate Risk Profile labels...
Evaluating Silhouette Scores:
 - K=3: Silhouette Score = 0.0787
 - K=4: Silhouette Score = 0.0739
 - K=5: Silhouette Score = 0.0763
-> Optimal K based on Silhouette Score: 3

Label Distribution setelah Clustering:
0    2103
1     831
2    2066
Name: count, dtype: int64
Cluster mapping: {np.int64(1): 0, np.int64(0): 1, np.int64(2): 2}

Labeled data saved to: archive/financial_risk_labeled.csv
